# Sub-Category: PGA-UNet vs Three Attention U-Net Variants (FracAtlas, R512)

> **Subset definition:** TOP-DICE and BOTTOM-DICE are defined **only once**, post hoc, from plain Attention U-Net Dice over the full test set. The other three models do not redefine image difficulty. PGA-UNet, Attention U-Net + Prompt Channel, and Attention U-Net + Prompt Crop are evaluated on the exact same top 50 and bottom 50 image stems.

> **Prompt condition:** the three prompt-guided models report both `center_zoom` and `center_shift` on the same subsets. Metrics are computed after image-level merging.

| Section | Content |
|---|---|
| Setup | Clone `main`, prepare FracAtlas, download four checkpoints |
| Attention U-Net | Rank the full test set and define TOP-DICE/BOTTOM-DICE; report N=50 per subset |
| PGA-UNet | Re-evaluate the same 50+50 stems |
| Prompt Channel | Re-evaluate the same 50+50 stems |
| Prompt Crop | Re-evaluate the same 50+50 stems |
| Summary | Four-model six-metric table, PGA Dice deltas, CSV and bar chart |


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 - SETUP (clone, dataset, checkpoints, utilities)
# ══════════════════════════════════════════════════════
%cd /content
import os, sys, cv2, json as _json
import numpy as np, torch
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle as Rect
!pip install -q gdown tqdm opencv-python matplotlib scipy
import gdown

REPO='https://github.com/ThongLuc2k3/PGA_Unet2D.git'
for d in ['PGA_Unet2D']:
    if not os.path.exists(f'/content/{d}'):
        os.system(f'git clone -q --branch main --single-branch {REPO} /content/{d}')
    print(f'  ✅ {d}')

if not os.path.exists('/content/dataset_FracAtlas'):
    gdown.download('https://drive.google.com/uc?id=1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv',
                   '/content/dataset_FracAtlas.zip', quiet=False)
    os.system('unzip -q /content/dataset_FracAtlas.zip -d /content/')
for d in ['PGA_Unet2D','PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation']:
    if not os.path.exists(f'/content/{d}/dataset_FracAtlas'):
        os.system(f'cp -r /content/dataset_FracAtlas /content/{d}/dataset_FracAtlas')
print('  ✅ Dataset ready')

os.makedirs('/content/checkpoints',exist_ok=True)
for cid, fn in [('1hlw7RlNgF0ChFIczSHuW4y3ygeUAmpYM', 'att_unet_best.pth'),
               ('TODO_CHECKPOINT_ID_pga512', 'pga_unet_center_mixed_x3_shift05_qhead_512_best.pth'),
               ('TODO_CHECKPOINT_ID_attunet_concat_prompt', 'attunet_concat_prompt_best.pth'),
               ('TODO_CHECKPOINT_ID_crop_attunet512', 'attunet_crop_best.pth')]:
    p=f'/content/checkpoints/{fn}'
    if not os.path.exists(p): gdown.download(f'https://drive.google.com/uc?id={cid}',p,quiet=False)
    print(f'  ✅ {fn}  {os.path.getsize(p)//1024}KB')

DEVICE,IMG_SIZE,ZOOM_R=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),512,0.30
IMG_DIR ='/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_FracAtlas/test/images'
MASK_DIR='/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_FracAtlas/test/masks'
JSON_DIR='/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_FracAtlas/test/annotations'
KEYS=['dice','iou','precision','recall','hd95','cbl']

def calc_metrics(prob,gt,eps=1e-6):
    pm=(prob>0.5).astype(np.float32); gm=(gt>0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p,g=pm.astype(bool),gm.astype(bool); hd95=float(IMG_SIZE)
    if p.any() and g.any():
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),iou=float((tp + eps) / (tp + fp + fn + eps)),
                precision=float(tp/(tp+fp+eps)),recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95,cbl=cbl,mask=pm)

def print_metrics(label,mlist):
    m={k:np.mean([r[k] for r in mlist]) for k in KEYS}
    print(f"  {label:<20} Dice={m['dice']:.4f} IoU={m['iou']:.4f} Pre={m['precision']:.4f}"
          f" Rec={m['recall']:.4f} HD95={m['hd95']:.2f} CBL={m['cbl']:.4f}")
    return m

def visualize(records, title, n=15, show_bbox=False):
    recs=records[:n]; nr=len(recs)
    fig,axes=plt.subplots(nr,4,figsize=(20,4.5*nr),squeeze=False)
    fig.suptitle(title,fontsize=13,fontweight='bold',y=1.005)
    for j,t in enumerate(['Input image','Prediction',
                           'Ground truth\n(Dice / IoU / CBL)',
                           'Overlap\n(Pre / Rec / HD95)']):
        axes[0,j].set_title(t,fontsize=9,fontweight='bold')
    for row,rec in enumerate(recs):
        gray=np.clip((rec['img_np']*0.5+0.5) * 255,0,255).astype(np.uint8)
        rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
        gt=rec['gt']; pm=rec['m']['mask']
        def ov(mask,clr,a=0.5):
            o=rgb.copy().astype(np.float32); o[mask>0.5]=o[mask>0.5]*(1-a)+np.array(clr)*a
            return np.clip(o,0,255).astype(np.uint8)
        diff=rgb.copy(); pb,gb=pm>0.5,gt>0.5
        diff[gb&~pb]=[30,180,30]; diff[pb&~gb]=[220,60,60]; diff[pb&gb]=[220,200,0]
        ax=axes[row,0]; ax.imshow(gray,cmap='gray')
        if show_bbox:
            for bx in rec.get('bboxes',[]):
                ax.add_patch(Rect((bx[0],bx[1]),bx[2]-bx[0],bx[3]-bx[1],lw=1.5,edgecolor='lime',facecolor='none'))
        ax.set_ylabel(rec['stem'],fontsize=7,rotation=0,labelpad=75,va='center'); ax.axis('off')
        axes[row,1].imshow(ov(pm,[30,120,255])); axes[row,1].axis('off')
        m=rec['m']
        axes[row,2].imshow(ov(gt,[30,180,30]))
        axes[row,2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                              fontsize=8,color='darkred',pad=2); axes[row,2].axis('off')
        axes[row,3].imshow(diff)
        axes[row,3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                              fontsize=8,color='saddlebrown',pad=2); axes[row,3].axis('off')
        for panel_ax in axes[row]:
            panel_ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=panel_ax.transAxes, fill=False,
                                             edgecolor='#555555', linewidth=1.0, clip_on=False))
    plt.tight_layout()
    fname=title.split('|')[0].strip()[:6].lower()+'_'+title.split('|')[-1].strip()[:4].lower()+'.png'
    plt.savefig(fname,dpi=100,bbox_inches='tight')
    export_qualitative_rows(fig, axes, recs); print(f'  ✅ {fname}')

class ImageMaskDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, mask_dir, img_size=512, is_train=False):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.is_train = is_train
        image_names = sorted(f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg")))
        mask_by_stem = {}
        for mask_name in sorted(os.listdir(mask_dir)):
            if not mask_name.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            stem = os.path.splitext(mask_name)[0]
            if stem in mask_by_stem:
                raise ValueError(f'Duplicate mask stem: {stem}')
            mask_by_stem[stem] = mask_name
        missing = [name for name in image_names if os.path.splitext(name)[0] not in mask_by_stem]
        if missing:
            raise FileNotFoundError(f'Missing masks for {len(missing)} images: {missing[:5]}')
        self.samples = [(name, mask_by_stem[os.path.splitext(name)[0]]) for name in image_names]
        self.images = [image_name for image_name, _ in self.samples]
        self.masks = [mask_name for _, mask_name in self.samples]

    def __len__(self):
        return len(self.samples)

    def _resize_and_pad(self, array, interpolation, pad_value=0):
        h, w = array.shape[:2]
        scale = min(self.img_size / w, self.img_size / h)
        new_w = max(1, int(round(w * scale)))
        new_h = max(1, int(round(h * scale)))
        resized = cv2.resize(array, (new_w, new_h), interpolation=interpolation)
        padded = np.full((self.img_size, self.img_size), pad_value, dtype=resized.dtype)
        pad_left = (self.img_size - new_w) // 2
        pad_top = (self.img_size - new_h) // 2
        padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
        return padded

    def __getitem__(self, idx):
        image_name, mask_name = self.samples[idx]
        image = cv2.imread(os.path.join(self.image_dir, image_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        image = self._resize_and_pad(image, cv2.INTER_LINEAR, pad_value=0)
        mask = self._resize_and_pad(mask, cv2.INTER_NEAREST, pad_value=0)
        image = (image.astype(np.float32) / 255.0 - 0.5) / 0.5
        mask = (mask > 127).astype(np.float32)
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return image, mask

# Load dataset & image stems
sys.path.insert(0,'/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation')
from qualitative_visualization import export_qualitative_rows
test_ds=ImageMaskDataset(image_dir=IMG_DIR, mask_dir=MASK_DIR, img_size=IMG_SIZE, is_train=False)
# The Attention U-Net dataset loader uses `os.listdir()` without sorting, so enforce sorting to keep `img_stems` aligned.
test_ds.images=sorted(test_ds.images)

img_stems=[os.path.splitext(f)[0] for f in test_ds.images]  # stems in the exact dataset order

all_raw=[]
for idx,(img_t,mask_t) in enumerate(DataLoader(test_ds,batch_size=1,shuffle=False)):
    all_raw.append({'idx':idx,
                    'img_np':img_t[0,0].numpy().copy(),   # [0,1]
                    'gt':mask_t[0,0].numpy().copy(),
                    'img_t':img_t.clone(),
                    'stem':img_stems[idx]})
SEC={}
print(f'\n✅ Setup complete | {len(all_raw)} samples | device={DEVICE}')
print(f'   Example stems: {img_stems[:3]} ...')


In [ ]:
from qualitative_visualization import export_qualitative_rows
# Redefine `visualize()` - TP = green / FP = red / FN = blue, with a legend
from matplotlib.patches import Patch, Rectangle as Rect
from IPython.display import display as _ipy_display

def visualize(records, title, n=15, show_bbox=False):
    recs = records[:n]; nr = len(recs)
    fig, axes = plt.subplots(nr, 4, figsize=(20, 4.5 * nr), squeeze=False)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.005)
    for j, t in enumerate(['Input image', 'Prediction',
                            'Ground truth\n(Dice / IoU / CBL)',
                            'TP/FP/FN\n(Pre / Rec / HD95)']):
        axes[0, j].set_title(t, fontsize=9, fontweight='bold')
    for row, rec in enumerate(recs):
        gray = np.clip((rec['img_np']*0.5+0.5) * 255, 0, 255).astype(np.uint8)
        rgb  = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        gt   = rec['gt']; pm = rec['m']['mask']

        def ov(mask, clr, a=0.5):
            o = rgb.copy().astype(np.float32)
            o[mask > 0.5] = o[mask > 0.5] * (1 - a) + np.array(clr) * a
            return np.clip(o, 0, 255).astype(np.uint8)

        # TP = green, FP = red, FN = blue (matched to the PGA convention)
        pb, gb = pm > 0.5, gt > 0.5
        bg_f = rgb.astype(np.float32) / 255.
        inter = bg_f.copy()
        inter[..., 1] = np.clip(inter[..., 1] + (pb & gb).astype(float)  * 0.9, 0, 1)
        inter[..., 0] = np.clip(inter[..., 0] + (pb & ~gb).astype(float) * 1.0, 0, 1)
        inter[..., 2] = np.clip(inter[..., 2] + (~pb & gb).astype(float) * 1.0, 0, 1)
        inter_u8 = (inter * 255).astype(np.uint8)

        ax = axes[row, 0]; ax.imshow(gray, cmap='gray')
        if show_bbox:
            for bx in rec.get('bboxes', []):
                ax.add_patch(Rect((bx[0], bx[1]), bx[2] - bx[0], bx[3] - bx[1],
                                  lw=1.5, edgecolor='lime', facecolor='none'))
        ax.set_ylabel(rec['stem'], fontsize=7, rotation=0, labelpad=75, va='center')
        ax.axis('off')
        axes[row, 1].imshow(ov(pm, [30, 120, 255])); axes[row, 1].axis('off')
        m = rec['m']
        axes[row, 2].imshow(ov(gt, [30, 180, 30]))
        axes[row, 2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                               fontsize=8, color='darkred', pad=2); axes[row, 2].axis('off')
        axes[row, 3].imshow(inter_u8)
        axes[row, 3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                               fontsize=8, color='saddlebrown', pad=2); axes[row, 3].axis('off')
        for panel_ax in axes[row]:
            panel_ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=panel_ax.transAxes, fill=False,
                                             edgecolor='#555555', linewidth=1.0, clip_on=False))
    fig.legend(
        handles=[Patch(facecolor='green', label='TP'),
                 Patch(facecolor='red',   label='FP'),
                 Patch(facecolor='blue',  label='FN')],
        loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.004)
    )
    plt.tight_layout()
    fname = title.split('|')[0].strip()[:6].lower() + '_' + title.split('|')[-1].strip()[:4].lower() + '.png'
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    export_qualitative_rows(fig, axes, recs)
    print(f'  ✅ {fname}')

print('✅ visualize() redefined - TP=green / FP=red / FN=blue + legend')

In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 0+1 - Attention U-Net
# Run on the full test set -> sort by Dice -> define TOP-DICE / BOTTOM-DICE stems
# (post hoc grouping by the Attention U-Net Dice scores, not by pre-assigned image difficulty)
# Metrics: N_METRIC=50 | Visualize: N_SHOW=10
# ══════════════════════════════════════════════════════
N_METRIC = 50   # number of images used for metric computation
N_SHOW = 10     # number of images displayed

%cd /content/PGA_Unet2D
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/PGA_Unet2D' not in sys.path:
    sys.path.insert(0, '/content/PGA_Unet2D')
else:
    sys.path.remove('/content/PGA_Unet2D'); sys.path.insert(0, '/content/PGA_Unet2D')

from models.networks.attention_unet_2D import Attention_UNet_2D
att_unet=Attention_UNet_2D(in_channels=1,n_classes=1).to(DEVICE)
att_unet.load_state_dict(torch.load('/content/checkpoints/att_unet_best.pth',map_location=DEVICE,weights_only=True))
att_unet.eval(); print('✅ Attention U-Net loaded')

all_att_unet_res=[]
with torch.no_grad():
    for s in tqdm(all_raw,'AttUNet-All'):
        prob=torch.sigmoid(att_unet(s['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        all_att_unet_res.append(calc_metrics(prob,s['gt']))
del att_unet

sorted_all=sorted(range(len(all_raw)),key=lambda i:all_att_unet_res[i]['dice'],reverse=True)
easy_idx=sorted_all[:N_METRIC]
hard_idx=sorted_all[-N_METRIC:][::-1]

easy_stems=[all_raw[i]['stem'] for i in easy_idx]   # TOP-DICE: images with the best Attention U-Net predictions
hard_stems =[all_raw[i]['stem'] for i in hard_idx]  # BOTTOM-DICE: images with the worst Attention U-Net predictions

def select_balanced_test_stems_from_all(stems, all_samples, n_multi=5, n_single=5):
    counts = {}
    for item in all_samples:
        counts[item['stem']] = counts.get(item['stem'], 0) + 1
    multi = [st for st in stems if counts.get(st, 0) >= 2]
    single = [st for st in stems if counts.get(st, 0) == 1]
    chosen = multi[:n_multi] + single[:n_single]
    if len(chosen) < (n_multi + n_single):
        chosen += [st for st in stems if st not in chosen][:(n_multi + n_single - len(chosen))]
    return chosen

top_dice_balanced_test_stems = select_balanced_test_stems_from_all(easy_stems, all_raw, 5, 5)
bottom_dice_balanced_test_stems = select_balanced_test_stems_from_all(hard_stems, all_raw, 5, 5)

print(f'\n  TOP-DICE top-{N_METRIC} (best Attention U-Net predictions)    Dice: {all_att_unet_res[easy_idx[0]]["dice"]:.4f} → {all_att_unet_res[easy_idx[-1]]["dice"]:.4f}')
print(f'  BOTTOM-DICE bottom-{N_METRIC} (worst Attention U-Net predictions)  Dice: {all_att_unet_res[hard_idx[0]]["dice"]:.4f} → {all_att_unet_res[hard_idx[-1]]["dice"]:.4f}')
print(f'\neasy_stems  (TOP-DICE)    = {easy_stems}')
print(f'hard_stems  (BOTTOM-DICE) = {hard_stems}')

bar='='*70
print(f'\n{bar}\n  SECTION 1 - Attention U-Net  (metrics N={N_METRIC}, show N={N_SHOW})\n{bar}')
SEC['att_unet']={
    'easy': print_metrics(f'TOP-DICE (N={N_METRIC})', [all_att_unet_res[i] for i in easy_idx]),
    'hard': print_metrics(f'BOTTOM-DICE (N={N_METRIC})', [all_att_unet_res[i] for i in hard_idx])
}

stem2idx={all_raw[i]['stem']:i for i in range(len(all_raw))}
recs_e=[dict(**all_raw[stem2idx[st]], m=all_att_unet_res[stem2idx[st]]) for st in top_dice_balanced_test_stems]
recs_h=[dict(**all_raw[stem2idx[st]], m=all_att_unet_res[stem2idx[st]]) for st in bottom_dice_balanced_test_stems]
visualize(recs_e, f'Attention U-Net | TOP-DICE - top-{N_SHOW} / {N_METRIC} images with the highest Dice (balanced qualitative stems)', n=N_SHOW)
visualize(recs_h, f'Attention U-Net | BOTTOM-DICE - bottom-{N_SHOW} / {N_METRIC} images with the lowest Dice (balanced qualitative stems)', n=N_SHOW)

In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 3 - PGA-UNet  (IMAGE-LEVEL merging)
# For each image: run all polygons, max-merge the probabilities, union the GT masks, then compute image-level metrics
# ══════════════════════════════════════════════════════

_EASY = []   # paste `easy_stems` from cell 2 when running this notebook independently
_HARD  = []
_N_METRIC, _N_SHOW = 50, 10
if 'easy_stems' not in dir() or not easy_stems: easy_stems, hard_stems = _EASY, _HARD
if 'N_METRIC'   not in dir(): N_METRIC = _N_METRIC
if 'N_SHOW'     not in dir(): N_SHOW   = _N_SHOW
assert easy_stems and hard_stems, "❌ Paste `easy_stems` / `hard_stems` from cell 2 into `_EASY` / `_HARD`."

%cd /content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation' not in sys.path:
    sys.path.insert(0, '/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation')
else:
    sys.path.remove('/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'); sys.path.insert(0, '/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation')

from models.networks.prompt_unet_2D import PGA_UNet
import importlib.util, torch
_spec=importlib.util.spec_from_file_location('pga_ds_mod','/content/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset.py')
_mod=importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
PGA_Dataset=_mod.PromptSegmentationDataset

pga=PGA_UNet(in_channels=1,n_classes=1,use_encoder_prompt=True).to(DEVICE)
pga.load_state_dict(torch.load('/content/checkpoints/pga_unet_center_mixed_x3_shift05_qhead_512_best.pth',
                                map_location=DEVICE,weights_only=True))
pga.eval(); print('✅ PGA-UNet loaded')

pga_ds=PGA_Dataset(image_dir=IMG_DIR, json_dir=JSON_DIR,
                   img_size=IMG_SIZE, is_train=False, prompt_mode='center_zoom')

stem_to_pga={}
for i,(img_name,_) in enumerate(pga_ds.all_samples):
    stem_to_pga.setdefault(os.path.splitext(img_name)[0],[]).append(i)


def calc_metrics_img(prob, gt, eps=1e-6, IMG_S=512):
    """Image-level evaluation: no LCC filtering (multiple polygons). Uses GT union and max-merged probabilities."""
    pm = (prob > 0.5).astype(np.float32)
    gm = (gt   > 0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p, g = pm.astype(bool), gm.astype(bool); hd95 = float(IMG_S)
    if p.any() and g.any():
        from scipy.ndimage import binary_erosion, distance_transform_edt
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp + eps) / (tp + fp + fn + eps)),
                precision=float(tp/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl, mask=pm)


def infer_pga_image(stem):
    """Run PGA on all polygons for a stem and return `img_np`, `gt_union`, `prob_max`, and `prompts`."""
    idxs = stem_to_pga.get(stem, [])
    if not idxs: return None
    gt_union = None; prob_max = None; prompts = []; img_np = None
    with torch.no_grad():
        for idx in idxs:
            img_t, gt_t, hm_t = pga_ds[idx]
            if img_np is None: img_np = img_t[0].numpy()
            gt_np = gt_t[0].numpy(); hm_np = hm_t[0].numpy()
            prob  = torch.sigmoid(pga(img_t.unsqueeze(0).to(DEVICE),
                                      hm_t.unsqueeze(0).to(DEVICE)))[0,0].cpu().numpy()
            gt_union = gt_np if gt_union is None else np.maximum(gt_union, gt_np)
            prob_max = prob  if prob_max is None else np.maximum(prob_max, prob)
            prompts.append(hm_np)
    return dict(stem=stem, img_np=img_np, gt=gt_union, prob=prob_max,
                prompts=prompts, n_samples=len(idxs))

print('  Running PGA inference per-image ...')
easy_img = [r for r in (infer_pga_image(s) for s in easy_stems) if r]
hard_img  = [r for r in (infer_pga_image(s) for s in hard_stems)  if r]
del pga

for r in easy_img + hard_img:
    r['m'] = calc_metrics_img(r['prob'], r['gt'])

bar='='*70
print(f'\n{bar}\n  SECTION 3 - PGA-UNet (image-level, N={N_METRIC} stems, using the same TOP-DICE/BOTTOM-DICE groups defined from Attention U-Net)\n{bar}')
SEC['pga'] = {
    'easy': print_metrics(f"TOP-DICE ({len(easy_img)} images)", [r['m'] for r in easy_img]),
    'hard': print_metrics(f"BOTTOM-DICE ({len(hard_img)} images)",  [r['m'] for r in hard_img]),
    'n_easy': len(easy_img), 'n_hard': len(hard_img),
}

from qualitative_visualization import export_qualitative_rows

def visualize_img(records, title, n=10):
    """Image-level vis: merged pred + GT union."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from IPython.display import display as _disp
    recs = records[:n]; nr = len(recs)
    if nr == 0: return
    fig, axes = plt.subplots(nr, 5, figsize=(20, 4*nr), squeeze=False)
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.002)
    for j,t in enumerate(['Input image','Prompts (merged)','Prediction (merged)','GT (union)','TP/FP/FN']):
        axes[0,j].set_title(t, fontsize=9, fontweight='bold')
    for row, rec in enumerate(recs):
        img_np = np.clip((rec['img_np']*0.5+0.5), 0, 1)
        gt     = (rec['gt']>0.5).astype(float)
        pred   = (rec['prob']>0.5).astype(float)
        pm_merged = np.max(np.stack(rec['prompts'],0),0) if rec.get('prompts') else np.zeros_like(img_np)
        tp=(pred*gt).sum(); fp=(pred*(1-gt)).sum(); fn=((1-pred)*gt).sum(); e=1e-6
        dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp + e) / (tp + fp + fn + e))
        pre=float(tp / (tp + fp+e));         rec_=float((tp+e)/(tp+fn+e))
        n_poly = rec.get('n_samples', '?')
        bg = np.stack([img_np]*3,-1)
        axes[row,0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        axes[row,0].set_ylabel(f"{rec['stem']} [{n_poly}p]\nDice={dice:.3f}", fontsize=7)
        axes[row,1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        axes[row,1].imshow(np.where(pm_merged>0,pm_merged,np.nan), cmap='hot', alpha=0.4, vmin=0, vmax=1)
        pr_ov=bg.copy(); pr_ov[...,0]=np.clip(pr_ov[...,0]+pred*0.55,0,1); pr_ov[...,1]=np.clip(pr_ov[...,1]-pred*0.2,0,1)
        axes[row,2].imshow(pr_ov); axes[row,2].set_title(f'Dice={dice:.3f} IoU={iou:.3f}',fontsize=7,color='darkred',pad=2)
        gt_ov=bg.copy(); gt_ov[...,1]=np.clip(gt_ov[...,1]+gt*0.55,0,1); gt_ov[...,0]=np.clip(gt_ov[...,0]-gt*0.2,0,1)
        axes[row,3].imshow(gt_ov)
        inter=bg.copy()
        inter[...,1]=np.clip(inter[...,1]+pred*gt*0.9,0,1)
        inter[...,0]=np.clip(inter[...,0]+pred*(1-gt)*1.0,0,1)
        inter[...,2]=np.clip(inter[...,2]+(1-pred)*gt*1.0,0,1)
        axes[row,4].imshow(inter); axes[row,4].set_title(f'Pre={pre:.3f} Rec={rec_:.3f}',fontsize=7,color='saddlebrown',pad=2)
        for ax in axes[row]:
            ax.axis('off')
            ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
    fig.legend(handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),Patch(facecolor='blue',label='FN')],
               loc='lower center',ncol=3,fontsize=8,bbox_to_anchor=(0.5,-0.004))
    plt.tight_layout()
    export_qualitative_rows(fig, axes, recs)

visualize_img(easy_img[:N_SHOW], f'PGA | TOP-DICE - first {N_SHOW} images (image-level merged, highest Attention U-Net Dice group)')
visualize_img(hard_img[:N_SHOW],  f'PGA | BOTTOM-DICE - first {N_SHOW} images (image-level merged, lowest Attention U-Net Dice group)')

---
## PGA-UNet center_shift on the shared subsets


In [ ]:
# ======================================================
# PGA-UNet center_shift on the same Attention U-Net-defined subsets
# The preceding PGA cell is the center_zoom result.
# ======================================================
SEC['pga_center_zoom'] = SEC.pop('pga')

pga = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
pga.load_state_dict(torch.load('/content/checkpoints/pga_unet_center_mixed_x3_shift05_qhead_512_best.pth',
                               map_location=DEVICE, weights_only=True))
pga.eval()
pga_ds = PGA_Dataset(image_dir=IMG_DIR, json_dir=JSON_DIR, img_size=IMG_SIZE,
                     is_train=False, prompt_mode='center_shift')
stem_to_pga = {}
for i, (img_name, _) in enumerate(pga_ds.all_samples):
    stem_to_pga.setdefault(os.path.splitext(img_name)[0], []).append(i)

easy_shift = [r for r in (infer_pga_image(s) for s in easy_stems) if r]
hard_shift = [r for r in (infer_pga_image(s) for s in hard_stems) if r]
for r in easy_shift + hard_shift:
    r['m'] = calc_metrics_img(r['prob'], r['gt'])
SEC['pga_center_shift'] = {
    'easy': print_metrics(f"TOP-DICE ({len(easy_shift)} images)", [r['m'] for r in easy_shift]),
    'hard': print_metrics(f"BOTTOM-DICE ({len(hard_shift)} images)", [r['m'] for r in hard_shift]),
    'n_easy': len(easy_shift), 'n_hard': len(hard_shift),
}
del pga
assert len(easy_shift) == len(easy_stems) and len(hard_shift) == len(hard_stems)
print('PGA-UNet center_shift completed on the exact shared top-50 and bottom-50 stems.')


---
## Attention U-Net + Prompt Channel on the shared subsets


In [ ]:
# ══════════════════════════════════════════════════════
# PART 3 - Attention U-Net + Prompt Channel Test (IMG_SIZE=512, 2 prompt modes)
# ══════════════════════════════════════════════════════
import os, sys, csv
import numpy as np, cv2, json as _json, glob
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE     = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'
if PGA_ROOT not in sys.path: sys.path.insert(0, PGA_ROOT)
for _k in list(sys.modules.keys()):
    if 'attunet_concat_prompt' in _k: del sys.modules[_k]
from models.networks.attunet_concat_prompt import AttUNetConcatPrompt

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE      = 512
SCALE_FACTOR  = 3.0
SHIFT_RATIO   = 0.5
BINARY_PROMPT = False  # Gaussian heatmap, matching concat-prompt-attunet-r512.ipynb
VARIANT_NAME  = 'Attention U-Net + prompt channel'
CKPT_PATH     = '/content/checkpoints/attunet_concat_prompt_best.pth'
os.makedirs(f'{PGA_ROOT}/results', exist_ok=True)


from dataset import PromptSegmentationDataset

class ConcatPromptDataset(PromptSegmentationDataset):
    """Dataset adapter that uses the exact PGA prompt pipeline from dataset.py."""
    def __init__(self, split='test', mode='center_zoom'):
        image_dir = f'{PGA_ROOT}/dataset_FracAtlas/{split}/images'
        json_dir = f'{PGA_ROOT}/dataset_FracAtlas/{split}/annotations'
        super().__init__(
            image_dir=image_dir,
            json_dir=json_dir,
            img_size=IMG_SIZE,
            is_train=(split == 'train'),
            prompt_mode=mode,
            scale_factor=SCALE_FACTOR,
            shift_ratio=SHIFT_RATIO,
            mixed_shift_prob=0.8,
        )
        # Preserve the metadata interface used by the image-level merge cells.
        self.samples = [
            (os.path.join(image_dir, img_name), shape_idx)
            for img_name, shape_idx in self.all_samples
        ]

    def __getitem__(self, idx):
        image, mask, prompt = super().__getitem__(idx)
        return image, prompt, mask


model = AttUNetConcatPrompt(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ Model loaded ({VARIANT_NAME})  device={DEVICE}')


def calc_hd95_c(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_metrics_img_c(prob_np, gt_np):
    pm = (prob_np > 0.5).astype(np.float32); gm = (gt_np > 0.5).astype(np.float32); eps = 1e-6
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    hd95 = calc_hd95_c(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0: cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        d = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2) / d, 0, 1))
    return dict(dice=float((2 * tp + eps) / (2 * tp + fp + fn + eps)), iou=float((tp + eps) / (tp + fp + fn + eps)),
                precision=float(tp / (tp + fp + eps)), recall=float((tp + eps) / (tp + fn + eps)),
                hd95=hd95, cbl=cbl)


KEYS  = ['dice', 'iou', 'precision', 'recall', 'hd95', 'cbl']
HDRS  = ['Dice↑', 'IoU↑', 'Prec↑', 'Rec↑', 'HD95↓', 'CBL↑']
MODES = ['center_zoom', 'center_shift']
concat_image_records = {}
concat_results = {}
concat_csv_rows = []

for mode in MODES:
    ds     = ConcatPromptDataset('test', mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=2)
    groups = OrderedDict()
    for i, (img_t, hm_t, gt_t) in enumerate(tqdm(loader, desc=f'[Concat {mode}]')):
        img_name  = os.path.basename(ds.samples[i][0])
        gt_np     = gt_t[0, 0].numpy()
        prompt_np = hm_t[0, 0].numpy()
        with torch.no_grad():
            prob = torch.sigmoid(model(img_t.to(DEVICE), hm_t.to(DEVICE)))[0, 0].cpu().numpy()
        if img_name not in groups:
            groups[img_name] = dict(img=img_t[0, 0].numpy(), gt_union=gt_np.copy(),
                                    prob_max=prob.copy(), prompts=[prompt_np])
        else:
            np.maximum(groups[img_name]['gt_union'], gt_np, out=groups[img_name]['gt_union'])
            np.maximum(groups[img_name]['prob_max'], prob, out=groups[img_name]['prob_max'])
            groups[img_name]['prompts'].append(prompt_np)
    img_recs = []
    for img_name in sorted(groups.keys()):
        g = groups[img_name]; m = calc_metrics_img_c(g['prob_max'], g['gt_union'])
        img_recs.append(dict(img_name=img_name, img=g['img'], gt=g['gt_union'],
                             prob=g['prob_max'], prompts=g['prompts'], n_samples=len(g['prompts']), **m))
    concat_image_records[mode] = img_recs
    m_avg = {k: np.mean([r[k] for r in img_recs]) for k in KEYS}
    concat_results[mode] = m_avg
    n_imgs = len(img_recs); n_samp = sum(r['n_samples'] for r in img_recs)
    concat_csv_rows.append([mode] + [f'{m_avg[k]:.4f}' for k in KEYS] + [str(n_imgs), str(n_samp)])

bar = '=' * 82
print(f'\n{bar}\n  {VARIANT_NAME} - Image-level metrics (GT union+max-merge)  IMG_SIZE={IMG_SIZE}\n{bar}')
print(f"  {'Mode':<16}" + ''.join(f'{h:>8}' for h in HDRS) + f"  {'N_img':>6}  {'N_smp':>6}")
print(f"  {'-'*78}")
for row in concat_csv_rows:
    print(f"  {row[0]:<16}" + ''.join(f'{row[i+1]:>8}' for i in range(len(KEYS))) + f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

with open(f'{PGA_ROOT}/results/attunet_concat_prompt_results.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['mode'] + KEYS + ['N_img', 'N_samples']); w.writerows(concat_csv_rows)
print(f'\n✅ CSV: {PGA_ROOT}/results/attunet_concat_prompt_results.csv')

---
## Attention U-Net + Prompt Crop on the shared subsets


In [ ]:
# ══════════════════════════════════════════════════════
# PART 4 - Attention U-Net + Prompt Crop Test (IMG_SIZE=512, 2 prompt modes)
# Predicts on the image cropped to the prompt box (not fed a heatmap channel),
# prediction pasted back into the full-image frame for evaluation. Ported
# from crop-prompt-attunet-r512.ipynb's run_eval, loading a saved checkpoint
# here instead of the freshly-trained in-memory model that file uses.
# ══════════════════════════════════════════════════════
import os, sys, csv
import cv2
import numpy as np
import torch
from matplotlib.patches import Rectangle
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE     = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512

for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset', 'models', 'attention_unet', 'unet')): del sys.modules[_k]
if PGA_ROOT not in sys.path: sys.path.insert(0, PGA_ROOT)
else: sys.path.remove(PGA_ROOT); sys.path.insert(0, PGA_ROOT)

from dataset import PromptSegmentationDataset
from models.networks.attention_unet_2D import Attention_UNet_2D

TEST_IMAGE_DIR = f'{PGA_ROOT}/dataset_FracAtlas/test/images'
TEST_JSON_DIR  = f'{PGA_ROOT}/dataset_FracAtlas/test/annotations'


class CroppedPromptDataset(PromptSegmentationDataset):
    """Same class as crop-prompt-attunet-r512.ipynb's CroppedPromptDataset,
    re-defined here so this cell is self-contained.
    """

    @staticmethod
    def _bbox_to_int_bounds(bx_min, by_min, bx_max, by_max, orig_h, orig_w):
        ix_min = int(np.floor(bx_min)); iy_min = int(np.floor(by_min))
        ix_max = int(np.ceil(bx_max));  iy_max = int(np.ceil(by_max))
        ix_min = max(0, min(ix_min, orig_w - 1)); iy_min = max(0, min(iy_min, orig_h - 1))
        ix_max = max(ix_min + 1, min(ix_max, orig_w)); iy_max = max(iy_min + 1, min(iy_max, orig_h))
        return ix_min, iy_min, ix_max, iy_max

    def __getitem__(self, idx):
        img_name, shape_idx = self.all_samples[idx]
        base = os.path.splitext(img_name)[0]
        image = cv2.imread(os.path.join(self.image_dir, img_name), cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = image.shape
        import json as _json
        with open(os.path.join(self.json_dir, base + '.json'), 'r', encoding='utf-8') as f:
            data = _json.load(f)
        points = np.array(data['shapes'][shape_idx]['points'])
        mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
        cv2.fillPoly(mask, [points.astype(np.int32)], 255)
        x_min, y_min = np.min(points, axis=0); x_max, y_max = np.max(points, axis=0)
        if self.prompt_mode == 'center_zoom':
            bx_min, bx_max, by_min, by_max = self._center_zoom_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)
        elif self.prompt_mode == 'center_shift':
            bx_min, bx_max, by_min, by_max = self._center_shift_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=idx)
        else:
            raise ValueError(f'Unknown prompt_mode: {self.prompt_mode}')
        ix_min, iy_min, ix_max, iy_max = self._bbox_to_int_bounds(bx_min, by_min, bx_max, by_max, orig_h, orig_w)
        image_crop = image[iy_min:iy_max, ix_min:ix_max]
        mask_crop = mask[iy_min:iy_max, ix_min:ix_max]
        image_crop = self._resize_and_pad(image_crop, cv2.INTER_LINEAR, pad_value=0)
        mask_crop = self._resize_and_pad(mask_crop, cv2.INTER_NEAREST, pad_value=0)
        image_crop = (image_crop.astype(np.float32) / 255.0 - 0.5) / 0.5
        mask_crop = (mask_crop > 127).astype(np.float32)
        image_t = torch.from_numpy(image_crop).unsqueeze(0)
        mask_t = torch.from_numpy(mask_crop).unsqueeze(0)
        mask_t = (mask_t > 0.5).float()
        bbox_t = torch.tensor([ix_min, iy_min, ix_max, iy_max], dtype=torch.long)
        orig_hw_t = torch.tensor([orig_h, orig_w], dtype=torch.long)
        return image_t, mask_t, bbox_t, orig_hw_t


def paste_prediction_back(pred_crop_bin, bbox, orig_h, orig_w, img_size):
    ix_min, iy_min, ix_max, iy_max = [int(v) for v in bbox]
    crop_w, crop_h = ix_max - ix_min, iy_max - iy_min
    scale_crop = min(img_size / crop_w, img_size / crop_h)
    new_w = max(1, int(round(crop_w * scale_crop))); new_h = max(1, int(round(crop_h * scale_crop)))
    pad_left, pad_top = (img_size - new_w) // 2, (img_size - new_h) // 2
    unpadded = pred_crop_bin[pad_top:pad_top + new_h, pad_left:pad_left + new_w]
    box_res = cv2.resize((unpadded * 255).astype(np.uint8), (crop_w, crop_h), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((orig_h, orig_w), dtype=np.uint8)
    canvas[iy_min:iy_max, ix_min:ix_max] = box_res
    scale_full = min(img_size / orig_w, img_size / orig_h)
    fw = max(1, int(round(orig_w * scale_full))); fh = max(1, int(round(orig_h * scale_full)))
    resized_full = cv2.resize(canvas, (fw, fh), interpolation=cv2.INTER_NEAREST)
    padded_full = np.zeros((img_size, img_size), dtype=np.uint8)
    pl, pt = (img_size - fw) // 2, (img_size - fh) // 2
    padded_full[pt:pt + fh, pl:pl + fw] = resized_full
    return (padded_full > 127).astype(np.float32)


def bbox_to_full_frame(bbox, orig_h, orig_w, img_size):
    ix_min, iy_min, ix_max, iy_max = [int(v) for v in bbox]
    scale_full = min(img_size / orig_w, img_size / orig_h)
    pl = (img_size - max(1, int(round(orig_w * scale_full)))) // 2
    pt = (img_size - max(1, int(round(orig_h * scale_full)))) // 2
    return (ix_min * scale_full + pl, iy_min * scale_full + pt, ix_max * scale_full + pl, iy_max * scale_full + pt)




def calc_hd95_p(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred); ge = gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))


def calc_cbl_p(pred_bin: np.ndarray, gt_bin: np.ndarray):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2)
    return float(np.clip(1.0 - d / gt_diag, 0.0, 1.0))


def dice_iou_pre_rec(pm, gm, smooth=1e-5):
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    pre = tp / (tp + fp + smooth)
    rec = (tp + smooth) / (tp + fn + smooth)
    return dice, iou, pre, rec


crop_model = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
crop_model.load_state_dict(torch.load('/content/checkpoints/attunet_crop_best.pth',
                                      map_location=DEVICE, weights_only=True))
crop_model.eval()
print(f'✅ Attention U-Net + prompt crop loaded  device={DEVICE}')


def run_eval_crop(prompt_mode):
    """Per-image-merged evaluation: every polygon's crop prediction is
    pasted back into shared full-image letterboxed space, then max-merged
    per image, exactly like crop-prompt-attunet-r512.ipynb's own run_eval.
    dice_crop is a simple mean of that image's per-polygon crop-frame Dice
    (no rigorous per-image merge exists for it, see that file's docstring).
    """
    crop_ds = CroppedPromptDataset(TEST_IMAGE_DIR, TEST_JSON_DIR, img_size=IMG_SIZE,
                                    is_train=False, prompt_mode=prompt_mode)
    full_ds = PromptSegmentationDataset(TEST_IMAGE_DIR, TEST_JSON_DIR, img_size=IMG_SIZE,
                                         is_train=False, prompt_mode=prompt_mode)
    crop_loader = DataLoader(crop_ds, batch_size=1, shuffle=False)
    full_loader = DataLoader(full_ds, batch_size=1, shuffle=False)

    groups = OrderedDict()
    with torch.no_grad():
        for i, ((img_crop, mask_crop, bbox, orig_hw), (img_full, mask_full, _)) in enumerate(
                zip(crop_loader, full_loader)):
            img_name = crop_ds.all_samples[i][0]
            out_crop = crop_model(img_crop.to(DEVICE))
            pred_crop = (torch.sigmoid(out_crop) > 0.5).float()[0, 0].cpu().numpy()
            gm_crop = mask_crop[0, 0].numpy()
            d_crop, _, _, _ = dice_iou_pre_rec(pred_crop, gm_crop)
            orig_h, orig_w = int(orig_hw[0, 0]), int(orig_hw[0, 1])
            pred_full = paste_prediction_back(pred_crop, bbox[0].tolist(), orig_h, orig_w, IMG_SIZE)
            gm_full = mask_full[0, 0].numpy()
            img_np = (img_full[0, 0].numpy() + 1) / 2.0
            box_full = bbox_to_full_frame(bbox[0].tolist(), orig_h, orig_w, IMG_SIZE)
            if img_name not in groups:
                groups[img_name] = dict(img=img_np, gt_union=gm_full.copy(), pred_union=pred_full.copy(),
                                         bboxes=[box_full], dice_crop_list=[d_crop])
            else:
                np.maximum(groups[img_name]['gt_union'], gm_full, out=groups[img_name]['gt_union'])
                np.maximum(groups[img_name]['pred_union'], pred_full, out=groups[img_name]['pred_union'])
                groups[img_name]['bboxes'].append(box_full)
                groups[img_name]['dice_crop_list'].append(d_crop)

    img_recs = []
    for img_name in sorted(groups.keys()):
        g = groups[img_name]
        d, i_, p, r = dice_iou_pre_rec(g['pred_union'], g['gt_union'])
        hd = calc_hd95_p(g['pred_union'].astype(bool), g['gt_union'].astype(bool))
        cbl = calc_cbl_p(g['pred_union'].astype(bool), g['gt_union'].astype(bool))
        img_recs.append(dict(
            img_name=img_name, img=g['img'], gt=g['gt_union'], pred=g['pred_union'], bboxes=g['bboxes'],
            dice_full=d, iou_full=i_, pre_full=p, rec_full=r, hd95_full=hd,
            cbl_full=cbl if cbl is not None else 0.0,
            dice_crop=float(np.mean(g['dice_crop_list'])), n_samples=len(g['dice_crop_list']),
        ))
    return {
        'dice_full': np.mean([rc['dice_full'] for rc in img_recs]),
        'iou_full':  np.mean([rc['iou_full'] for rc in img_recs]),
        'pre_full':  np.mean([rc['pre_full'] for rc in img_recs]),
        'rec_full':  np.mean([rc['rec_full'] for rc in img_recs]),
        'hd95_full': np.mean([rc['hd95_full'] for rc in img_recs]),
        'cbl_full':  np.mean([rc['cbl_full'] for rc in img_recs]),
        'dice_crop': np.mean([rc['dice_crop'] for rc in img_recs]),
        'n_img':     len(img_recs),
        'n_samples': sum(rc['n_samples'] for rc in img_recs),
        'img_recs':  img_recs,
    }


print('\n' + '=' * 90)
print('PART 4 - SUBSET-SOURCE EVALUATION - ATTENTION U-NET + PROMPT CROP (per-image-merged)')
print('=' * 90)
crop_scenarios = {'center_zoom': 'center_zoom', 'center_shift': 'center_shift'}
crop_results = {}
for name, mode in crop_scenarios.items():
    crop_results[name] = run_eval_crop(mode)
    print(f"[{name}] {crop_results[name]['n_img']} images ({crop_results[name]['n_samples']} polygon samples), completed.")

crop_header = (f"\n{'Scenario':<12} {'DiceFull':>9} {'IoUFull':>9} {'PreFull':>9} "
               f"{'RecFull':>9} {'HD95Full':>10} {'CBLFull':>9} {'DiceCrop':>9} {'N_img':>6} {'N_smp':>6}")
print(crop_header)
print('-' * 96)
for name, r in crop_results.items():
    print(f"{name:<12} {r['dice_full']:>9.4f} {r['iou_full']:>9.4f} {r['pre_full']:>9.4f} "
          f"{r['rec_full']:>9.4f} {r['hd95_full']:>10.2f} {r['cbl_full']:>9.4f} {r['dice_crop']:>9.4f} "
          f"{r['n_img']:>6} {r['n_samples']:>6}")

os.makedirs(f'{PGA_ROOT}/results', exist_ok=True)
with open(f'{PGA_ROOT}/results/attunet_crop_prompt_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['model', 'scenario', 'dice_full', 'iou_full', 'precision_full',
                      'recall_full', 'hd95_full', 'cbl_full', 'dice_crop', 'n_img', 'n_samples'])
    for name, r in crop_results.items():
        writer.writerow(['AttUNet2D_CropPrompt', name, f"{r['dice_full']:.4f}", f"{r['iou_full']:.4f}",
                          f"{r['pre_full']:.4f}", f"{r['rec_full']:.4f}", f"{r['hd95_full']:.4f}",
                          f"{r['cbl_full']:.4f}", f"{r['dice_crop']:.4f}", r['n_img'], r['n_samples']])
print(f"\n✅ CSV: {PGA_ROOT}/results/attunet_crop_prompt_results.csv")


---
## Collect the shared TOP-DICE/BOTTOM-DICE subset results


In [ ]:
# ======================================================
# SUBSET EVALUATION - prompt-matched Attention U-Net variants
# TOP-DICE and BOTTOM-DICE remain defined only by plain Attention U-Net.
# Both prompt variants are evaluated on exactly those same image stems,
# separately for center_zoom and center_shift.
# ======================================================
def subset_records(records, stems):
    by_stem = {os.path.splitext(r['img_name'])[0]: r for r in records}
    return [by_stem[s] for s in stems if s in by_stem]

for scenario in ['center_zoom', 'center_shift']:
    easy = subset_records(concat_image_records[scenario], easy_stems)
    hard = subset_records(concat_image_records[scenario], hard_stems)
    SEC[f'attunet_channel_{scenario}'] = {
        'easy': print_metrics(f"{scenario} TOP-DICE ({len(easy)} images)", easy),
        'hard': print_metrics(f"{scenario} BOTTOM-DICE ({len(hard)} images)", hard),
        'n_easy': len(easy), 'n_hard': len(hard),
    }
    assert len(easy) == len(easy_stems) and len(hard) == len(hard_stems)

def normalize_crop_metric(r):
    return dict(dice=float(r['dice_full']), iou=float(r['iou_full']),
                precision=float(r['pre_full']), recall=float(r['rec_full']),
                hd95=float(r['hd95_full']), cbl=float(r['cbl_full']))

for scenario in ['center_zoom', 'center_shift']:
    easy_raw = subset_records(crop_raw_results[scenario]['img_recs'], easy_stems)
    hard_raw = subset_records(crop_raw_results[scenario]['img_recs'], hard_stems)
    easy = [normalize_crop_metric(r) for r in easy_raw]
    hard = [normalize_crop_metric(r) for r in hard_raw]
    SEC[f'attunet_crop_{scenario}'] = {
        'easy': print_metrics(f"{scenario} TOP-DICE ({len(easy)} images)", easy),
        'hard': print_metrics(f"{scenario} BOTTOM-DICE ({len(hard)} images)", hard),
        'n_easy': len(easy), 'n_hard': len(hard),
    }
    assert len(easy) == len(easy_stems) and len(hard) == len(hard_stems)
print('All prompt variants and both scenarios used the exact Attention U-Net-defined top-50 and bottom-50 stems.')


---
## Summary and comparison


In [ ]:
# ======================================================
# SUMMARY - seven model/scenario rows on the same Attention U-Net-defined subsets
# ======================================================
import csv, os
GRP_LABEL = {'easy': 'TOP-DICE', 'hard': 'BOTTOM-DICE'}
N_EASY, N_HARD = len(easy_stems), len(hard_stems)
MODEL_ROWS = [
    ('Attention U-Net', 'no_prompt', 'att_unet'),
    ('AttUNet + Prompt Channel', 'center_zoom', 'attunet_channel_center_zoom'),
    ('AttUNet + Prompt Channel', 'center_shift', 'attunet_channel_center_shift'),
    ('AttUNet + Prompt Crop', 'center_zoom', 'attunet_crop_center_zoom'),
    ('AttUNet + Prompt Crop', 'center_shift', 'attunet_crop_center_shift'),
    ('PGA-UNet', 'center_zoom', 'pga_center_zoom'),
    ('PGA-UNet', 'center_shift', 'pga_center_shift'),
]
bar = '=' * 125
print(f'\n{bar}')
print(f'  SUMMARY | {N_EASY} TOP-DICE + {N_HARD} BOTTOM-DICE images, defined post hoc only by plain Attention U-Net')
print(bar)
print(f"  {'Model':<27} {'Scenario':<13} {'Subset':<12} {'N':>5} {'Dice':>8} {'IoU':>8} {'Pre':>8} {'Rec':>8} {'HD95':>9} {'CBL':>8}")
print('  ' + '-' * 113)
csv_rows = []
for model, scenario, key in MODEL_ROWS:
    for grp in ['easy', 'hard']:
        m = SEC[key][grp]
        n = SEC[key].get('n_easy' if grp == 'easy' else 'n_hard', N_EASY if grp == 'easy' else N_HARD)
        print(f"  {model:<27} {scenario:<13} {GRP_LABEL[grp]:<12} {n:>5}"
              f" {m['dice']:>8.4f} {m['iou']:>8.4f} {m['precision']:>8.4f}"
              f" {m['recall']:>8.4f} {m['hd95']:>9.2f} {m['cbl']:>8.4f}")
        csv_rows.append([model, scenario, GRP_LABEL[grp], n] + [f"{m[k]:.4f}" for k in KEYS])
    print('  ' + '-' * 113)

print('\nDice delta of PGA-UNet against prompt-matched variants under the same scenario:')
for scenario in ['center_zoom', 'center_shift']:
    pga_key = f'pga_{scenario}'
    for name, base_key in [('Prompt Channel', f'attunet_channel_{scenario}'),
                           ('Prompt Crop', f'attunet_crop_{scenario}')]:
        print(f"  {scenario:<13} vs {name:<15}"
              f" TOP={SEC[pga_key]['easy']['dice']-SEC[base_key]['easy']['dice']:+.4f}"
              f" BOTTOM={SEC[pga_key]['hard']['dice']-SEC[base_key]['hard']['dice']:+.4f}")
print(bar)

os.makedirs('results', exist_ok=True)
csv_path = 'results/subcat_pga_vs_attunet_variants_r512.csv'
with open(csv_path, 'w', newline='') as f:
    csv.writer(f).writerows([['model', 'scenario', 'group', 'N'] + KEYS] + csv_rows)
print(f'Saved CSV: {csv_path}')

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
colors = ['#6699cc', '#66bb6a', '#2e7d32', '#ffa726', '#ef6c00', '#ef5350', '#b71c1c']
for gi, grp in enumerate(['easy', 'hard']):
    labels = [f'{m}\n{s}' for m, s, _ in MODEL_ROWS]
    vals = [SEC[key][grp]['dice'] for _, _, key in MODEL_ROWS]
    bars = axes[gi].bar(np.arange(len(vals)), vals, color=colors, edgecolor='white')
    for b, v in zip(bars, vals):
        axes[gi].text(b.get_x()+b.get_width()/2, v+0.012, f'{v:.3f}', ha='center', fontsize=8)
    axes[gi].set_xticks(np.arange(len(labels))); axes[gi].set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    axes[gi].set_title(f'{GRP_LABEL[grp]} ({N_EASY if grp=="easy" else N_HARD} images)')
    axes[gi].grid(axis='y', alpha=0.3); axes[gi].set_ylim(0, 1.15)
axes[0].set_ylabel('Dice')
fig.suptitle('Attention U-Net-defined Top/Bottom-Dice subsets: all models and prompt scenarios', fontweight='bold')
plt.tight_layout()
plot_path = 'results/subcat_attunet_variants_r512_bar.png'
plt.savefig(plot_path, dpi=130, bbox_inches='tight'); plt.show()
print(f'Saved plot: {plot_path}')
